In [1]:
import pandas as pd
import os
import numpy as np

In [2]:
def get_aa_counts(selected_aligned_seq):
    AA_LIST = list("ACDEFGHIKLMNPQRSTVWYO-")
    N_AA = len(AA_LIST)
    aa_to_idx = {aa: i for i, aa in enumerate(AA_LIST)}
    idx_to_aa = {v: k for k, v in aa_to_idx.items()}
    aa_idx_array = np.vectorize(aa_to_idx.get, otypes=[int])(selected_aligned_seq)
    alignments = np.zeros([aa_idx_array.shape[1], N_AA])
    for i in range(aa_idx_array.shape[1]):
        unique, counts = np.unique(aa_idx_array[:,i], return_counts=True)
        alignments[i, unique] = counts
    return alignments

In [17]:
root_dir = "/scratch/alpine/fana6609/ML/PLM-Epistasis"
data_dir = "data"
info_df = pd.read_csv(os.path.join(root_dir, data_dir, "input_VRC01_IC80.csv"))
aligned_sequences = np.array([list(seq) for seq in info_df['Sequence']])
resno_df = pd.read_csv(os.path.join(root_dir, data_dir, "selected_residues_IC80.csv"))
resno_array = resno_df["ResLabel"].values

AA_LIST = list("ACDEFGHIKLMNPQRSTVWYO-")
N_AA = len(AA_LIST)
aa_to_idx = {aa: i for i, aa in enumerate(AA_LIST)}
idx_to_aa = {v: k for k, v in aa_to_idx.items()}

all_labels = info_df["Label"].values
class_0_mask = (all_labels == 0)
aa_counts_0 = get_aa_counts(aligned_sequences[class_0_mask]).astype(int)
class_0_df = pd.DataFrame(aa_counts_0, index=resno_array, columns=AA_LIST)

class_1_mask = (all_labels == 1)
aa_counts_1 = get_aa_counts(aligned_sequences[class_1_mask]).astype(int)
class_1_df = pd.DataFrame(aa_counts_1, index=resno_array, columns=AA_LIST)

assert class_0_df.shape == class_1_df.shape

print("Input Sequence Length:", class_0_df.shape[0])

class_0_df.to_csv(os.path.join(root_dir, data_dir, "class_0_aa_counts.csv"))
class_1_df.to_csv(os.path.join(root_dir, data_dir, "class_1_aa_counts.csv"))

Input Sequence Length: 209
